In [ ]:
import pandas as pd
import numpy as np
import logging

# ============================
# PARAMETERS
# ============================

SIMULATIONS = 1000
SHORTAGE_PENALTY = 10
HOLDING_PENALTY = 1
AVAILABLE_HOURS = 22

file_path = "C:/Users/Ex0164/Book1.xlsx"
daily_path = "C:/Users/Ex0164/Important codes/Child_for_26feb_actual.xlsx"
compatibility_path = "C:/Users/Ex0164/Important codes/compatibility_matrix.xlsx"

logging.basicConfig(level=logging.INFO, format="%(levelname)s: %(message)s")

# ============================
# LOAD FILES
# ============================

stats = pd.read_excel(file_path, sheet_name="Sheet2")
daily = pd.read_excel(daily_path, sheet_name="Sheet1")

compatibility = pd.read_excel(compatibility_path)

data = stats.merge(daily, on="Material")

# ============================
# PRODUCTION RATE
# ============================

data["Rate"] = (3600 / data["Cycle time "]) * data["cavity"]

# ============================
# MONTE CARLO OPTIMIZER
# ============================

def optimize_part(row):

    inventory = row["Inventory on 24th"]
    demand_tomorrow = row["2026-02-27 Total Production Plan"]
    tentative_future = row["2026-03-01 Total Production Plan"]
    std = row["Std_Deviation"]
    rate = row["Rate"]

    best_qty = 0
    best_cost = np.inf

    max_qty = row["Feb INDENT"] * 1.2

    for qty in np.arange(0, max_qty, 50):

        future_stock = inventory + qty - demand_tomorrow

        simulated_demand = np.random.normal(
            tentative_future, std, SIMULATIONS
        )

        shortage = np.maximum(0, simulated_demand - future_stock)

        expected_shortage = shortage.mean()
        expected_inventory = max(0, future_stock - simulated_demand.mean())

        cost = (
            SHORTAGE_PENALTY * expected_shortage +
            HOLDING_PENALTY * expected_inventory
        )

        hours_needed = qty / rate

        if hours_needed <= AVAILABLE_HOURS and cost < best_cost:

            best_cost = cost
            best_qty = qty

    return best_qty, best_cost

# ============================
# MACHINE ASSIGNMENT
# ============================

machine_hours = {}

final_plan = []

for _, row in data.iterrows():

    part = row["Material"]

    planned_qty, cost = optimize_part(row)

    if planned_qty == 0:
        continue

    rate = row["Rate"]
    hours = planned_qty / rate

    # compatible machines
    comp_row = compatibility[compatibility["Part"] == part]

    if comp_row.empty:
        logging.info(f"No machine compatibility found for {part}")
        continue

    machines = comp_row.drop(columns=["Part"]).columns

    assigned = False

    for machine in machines:

        if comp_row.iloc[0][machine] == 1:

            used_hours = machine_hours.get(machine, 0)

            if used_hours + hours <= AVAILABLE_HOURS:

                machine_hours[machine] = used_hours + hours

                final_plan.append({
                    "Part": part,
                    "Machine": machine,
                    "Production_Qty": planned_qty,
                    "Run_Hours": round(hours,2),
                    "Rate": round(rate,2),
                    "Cost": round(cost,2)
                })

                assigned = True
                break

    if not assigned:

        logging.info(f"{part} not scheduled (machines full)")

# ============================
# SAVE OUTPUT
# ============================

final_df = pd.DataFrame(final_plan)

final_df.to_excel("APS_MonteCarlo_CompatibilityPlan.xlsx", index=False)

logging.info("Planning Complete")

In [ ]:
import pandas as pd
import numpy as np
import logging

# ============================
# PARAMETERS
# ============================

SIMULATIONS = 1000
SHORTAGE_PENALTY = 10
HOLDING_PENALTY = 1
AVAILABLE_HOURS = 22

book_path = "C:/Users/Ex0164/Book1.xlsx"
daily_path = "C:/Users/Ex0164/Important codes/Child_for_26feb_actual.xlsx"
matrix_path = "C:/Users/Ex0164/Important codes/compatibility_matrix.xlsx"

logging.basicConfig(level=logging.INFO, format="%(levelname)s: %(message)s")

# ============================
# LOAD DATA
# ============================

hz_parts = pd.read_excel(book_path, sheet_name="HZ")
vt_parts = pd.read_excel(book_path, sheet_name="VT")

stats = pd.read_excel(book_path, sheet_name="Sheet2")
daily = pd.read_excel(daily_path, sheet_name="Sheet1")

hz_matrix = pd.read_excel(matrix_path, sheet_name="HZ_Matrix")
vt_matrix = pd.read_excel(matrix_path, sheet_name="VT_Matrix")

# ============================
# MERGE DATA
# ============================

data = stats.merge(daily, on="Material")

# ============================
# PRODUCTION RATE
# ============================

data["Rate"] = (3600 / data["Cycle time "]) * data["cavity"]

# ============================
# MONTE CARLO OPTIMIZER
# ============================

def optimize_part(row):

    inventory = row["Inventory on 24th"]
    demand_tomorrow = row["2026-02-27 Total Production Plan"]
    tentative_future = row["2026-03-01 Total Production Plan"]
    std = row["Std_Deviation"]
    rate = row["Rate"]

    best_qty = 0
    best_cost = np.inf

    max_qty = row["Feb INDENT"] * 1.2

    for qty in np.arange(0, max_qty, 50):

        future_stock = inventory + qty - demand_tomorrow

        simulated_demand = np.random.normal(
            tentative_future, std, SIMULATIONS
        )

        shortage = np.maximum(0, simulated_demand - future_stock)

        expected_shortage = shortage.mean()
        expected_inventory = max(0, future_stock - simulated_demand.mean())

        cost = (
            SHORTAGE_PENALTY * expected_shortage +
            HOLDING_PENALTY * expected_inventory
        )

        hours_needed = qty / rate

        if hours_needed <= AVAILABLE_HOURS and cost < best_cost:

            best_cost = cost
            best_qty = qty

    return best_qty, best_cost

# ============================
# MACHINE ASSIGNMENT FUNCTION
# ============================

def assign_machines(data, matrix):

    machine_hours = {}
    final_plan = []

    for _, row in data.iterrows():

        part = row["Material"]

        planned_qty, cost = optimize_part(row)

        if planned_qty == 0:
            continue

        rate = row["Rate"]
        hours = planned_qty / rate

        comp_row = matrix[matrix["Part"] == part]

        if comp_row.empty:
            logging.info(f"No machine found for {part}")
            continue

        machines = comp_row.columns[1:]

        assigned = False

        for machine in machines:

            if comp_row.iloc[0][machine] == 1:

                used_hours = machine_hours.get(machine, 0)

                if used_hours + hours <= AVAILABLE_HOURS:

                    machine_hours[machine] = used_hours + hours

                    final_plan.append({
                        "Part": part,
                        "Machine": machine,
                        "Production_Qty": planned_qty,
                        "Run_Hours": round(hours,2),
                        "Rate": round(rate,2),
                        "Cost": round(cost,2)
                    })

                    assigned = True
                    break

        if not assigned:

            logging.info(f"{part} not scheduled (machines full)")

    return pd.DataFrame(final_plan)

# ============================
# FILTER PARTS BY FAMILY
# ============================

hz_data = data[data["Material"].isin(hz_parts["Part"])]
vt_data = data[data["Material"].isin(vt_parts["Part"])]

# ============================
# RUN APS
# ============================

hz_plan = assign_machines(hz_data, hz_matrix)
vt_plan = assign_machines(vt_data, vt_matrix)

# ============================
# SAVE OUTPUT
# ============================

with pd.ExcelWriter("APS_MonteCarlo_CompatibilityPlan.xlsx") as writer:

    hz_plan.to_excel(writer, sheet_name="HZ_Plan", index=False)
    vt_plan.to_excel(writer, sheet_name="VT_Plan", index=False)

logging.info("Planning Complete")

In [ ]:
import pandas as pd
import numpy as np
import logging

# ============================
# PARAMETERS
# ============================

SIMULATIONS = 1000
SHORTAGE_PENALTY = 10
HOLDING_PENALTY = 1
AVAILABLE_HOURS = 22

MIN_RUN_HOURS = 4
MIN_PART_QTY = 150

book_path = "C:/Users/Ex0164/Book1.xlsx"
daily_path = "C:/Users/Ex0164/Important codes/Child_for_26feb_actual.xlsx"
matrix_path = "C:/Users/Ex0164/Important codes/compatibility_matrix.xlsx"

logging.basicConfig(level=logging.INFO, format="%(levelname)s: %(message)s")

# ============================
# LOAD DATA
# ============================

hz_parts = pd.read_excel(book_path, sheet_name="HZ")
vt_parts = pd.read_excel(book_path, sheet_name="VT")

stats = pd.read_excel(book_path, sheet_name="Sheet2")
daily = pd.read_excel(daily_path, sheet_name="Sheet1")

hz_matrix = pd.read_excel(matrix_path, sheet_name="HZ_Matrix")
vt_matrix = pd.read_excel(matrix_path, sheet_name="VT_Matrix")

data = stats.merge(daily, on="Material")

# ============================
# RATE
# ============================

data["Rate"] = (3600 / data["Cycle time "]) * data["cavity"]

# ============================
# MONTE CARLO OPTIMIZER
# ============================

def optimize_part(row):

    inventory = row["Inventory on 24th"]
    demand_tomorrow = row["2026-02-27 Total Production Plan"]
    tentative_future = row["2026-03-01 Total Production Plan"]
    std = row["Std_Deviation"]
    rate = row["Rate"]

    best_qty = 0
    best_cost = np.inf

    max_qty = row["Feb INDENT"] * 1.2

    for qty in np.arange(0, max_qty, 50):

        future_stock = inventory + qty - demand_tomorrow

        simulated_demand = np.random.normal(
            tentative_future, std, SIMULATIONS
        )

        shortage = np.maximum(0, simulated_demand - future_stock)

        expected_shortage = shortage.mean()
        expected_inventory = max(0, future_stock - simulated_demand.mean())

        cost = (
            SHORTAGE_PENALTY * expected_shortage +
            HOLDING_PENALTY * expected_inventory
        )

        hours_needed = qty / rate

        if hours_needed <= AVAILABLE_HOURS and cost < best_cost:
            best_cost = cost
            best_qty = qty

    return best_qty, best_cost

# ============================
# APS ENGINE
# ============================

def run_planner(data, matrix):

    machine_hours = {}
    final_plan = []
    not_planned = []

    for _, row in data.iterrows():

        part = row["Material"]

        demand = row["2026-02-27 Total Production Plan"]
        indent = row["Feb INDENT"]
        inventory = row["Inventory on 24th"]

        # ------------------
        # SMALL PART FILTER
        # ------------------

        if demand < MIN_PART_QTY or indent < MIN_PART_QTY:

            not_planned.append({
                "Part": part,
                "Reason": "Small demand/indent (<150)"
            })

            continue

        planned_qty, cost = optimize_part(row)

        if planned_qty == 0:

            not_planned.append({
                "Part": part,
                "Reason": "Optimizer suggested zero production"
            })

            continue

        rate = row["Rate"]
        hours = planned_qty / rate

        if hours < MIN_RUN_HOURS:

            hours = MIN_RUN_HOURS
            planned_qty = rate * MIN_RUN_HOURS

        comp_row = matrix[matrix["Part"] == part]

        if comp_row.empty:

            not_planned.append({
                "Part": part,
                "Reason": "No compatible machine"
            })

            continue

        machines = comp_row.columns[1:]

        assigned = False

        for machine in machines:

            if comp_row.iloc[0][machine] == 1:

                used = machine_hours.get(machine, 0)
                free = AVAILABLE_HOURS - used

                if free >= hours:

                    machine_hours[machine] = used + hours

                    final_plan.append({
                        "Part": part,
                        "Machine": machine,
                        "Run_Hours": round(hours,2),
                        "Production_Qty": round(planned_qty,0),
                        "Cost": round(cost,2)
                    })

                    assigned = True
                    break

                # -------------------------
                # PARTIAL PRODUCTION OPTION
                # -------------------------

                elif free > MIN_RUN_HOURS:

                    machine_hours[machine] = AVAILABLE_HOURS

                    partial_qty = free * rate

                    final_plan.append({
                        "Part": part,
                        "Machine": machine,
                        "Run_Hours": round(free,2),
                        "Production_Qty": round(partial_qty,0),
                        "Cost": round(cost,2)
                    })

                    assigned = True
                    break

        if not assigned:

            if inventory >= demand:

                reason = "Inventory sufficient for today"

            else:

                shortage = demand - inventory
                reason = f"Machine capacity full (shortage risk {shortage})"

            not_planned.append({
                "Part": part,
                "Inventory": inventory,
                "Demand": demand,
                "Reason": reason
            })

    return pd.DataFrame(final_plan), pd.DataFrame(not_planned)

# ============================
# FILTER HZ / VT
# ============================

hz_data = data[data["Material"].isin(hz_parts["Part"])]
vt_data = data[data["Material"].isin(vt_parts["Part"])]

# ============================
# RUN PLANNER
# ============================

hz_plan, hz_not = run_planner(hz_data, hz_matrix)
vt_plan, vt_not = run_planner(vt_data, vt_matrix)

# ============================
# SAVE OUTPUT
# ============================

with pd.ExcelWriter("APS_Final_Plan.xlsx") as writer:

    hz_plan.to_excel(writer, sheet_name="HZ_Plan", index=False)
    vt_plan.to_excel(writer, sheet_name="VT_Plan", index=False)

    hz_not.to_excel(writer, sheet_name="HZ_Not_Planned", index=False)
    vt_not.to_excel(writer, sheet_name="VT_Not_Planned", index=False)

logging.info("Planning Complete")